# 👁️ Optimized Vision Transformer Training on CIFAR-10

This notebook compares two Vision Transformer (ViT) training implementations on CIFAR-10:

1. **Option A (vmap-based)**: Your original single-image model design, batch-vectorized using `torch.func.vmap`, with DataLoader & AMP (bfloat16) optimizations.
2. **Option B (Standard Batched + FlashAttention)**: A fully refactored, native batched model using PyTorch's native `scaled_dot_product_attention` (SDPA), Automatic Mixed Precision (AMP), TensorFloat-32 (TF32), and `torch.compile`.

In [1]:
import os
import sys
from pathlib import Path
# Add project root to sys.path so we can import from src
sys.path.append(str(Path.cwd().parent))

import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.func import functional_call
import matplotlib.pyplot as plt
import numpy as np

from src.utils import get_cifar_dataloaders
from src.models_vmap import VisualTransformer as VmapVisualTransformer
from src.models_batched import BatchedVisualTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Load Data

We load the CIFAR-10 dataset using the shared utility. We use a batch size of 512, 4 worker threads, and pin memory to maximize GPU data-transfer speed.

In [2]:
# Batch size of 512 to fully saturate the GPU
train_loader, test_loader, testset = get_cifar_dataloaders(
    batch_size=512, 
    num_workers=4, 
    pin_memory=True,
    train_subset_size=10000, 
    test_subset_size=1000
)

## 2. Train Option A: Optimized `vmap` (Single-Image Model)

We train the original single-image architecture scaled with `torch.func.vmap`. Optimizations applied:
* Batch size scaled to 512
* `num_workers=4`, `pin_memory=True`
* Fused AdamW optimizer
* Automatic Mixed Precision (AMP) in `bfloat16`
* `zero_grad(set_to_none=True)`

In [3]:
model_vmap = VmapVisualTransformer(
    img_size=32, patch_size=4, in_channels=3, num_classes=10,
    embed_dim=192, depth=6, num_heads=6, head_dim=32, mlp_ratio=2.0
).to(device)

params = dict(model_vmap.named_parameters())
buffers = dict(model_vmap.named_buffers())

def forward_fn(params, buffers, x):
    return functional_call(model_vmap, (params, buffers), x)
    
batched_forward = torch.func.vmap(forward_fn, in_dims=(None, None, 0))

# Fused AdamW for faster updates on GPU
optimizer = optim.AdamW(params.values(), lr=3.0e-3, weight_decay=5e-2, fused=True)
epochs = 10
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
criterion = nn.CrossEntropyLoss()

print("Training Option A (Optimized vmap)...")
start_vmap = time.time()

for epoch in range(epochs):
    model_vmap.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = batched_forward(params, buffers, inputs)
            loss = criterion(outputs, targets)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    scheduler.step()
    train_acc = 100. * correct / total
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")

vmap_total_time = time.time() - start_vmap
print(f"Option A finished in {vmap_total_time:.2f}s")

Training Option A (Optimized vmap)...
Epoch 01/10 | Loss: 2.1868 | Train Acc: 17.87%
Epoch 02/10 | Loss: 2.0804 | Train Acc: 20.87%
Epoch 03/10 | Loss: 2.0768 | Train Acc: 20.90%
Epoch 04/10 | Loss: 2.0387 | Train Acc: 21.84%
Epoch 05/10 | Loss: 2.0043 | Train Acc: 23.51%
Epoch 06/10 | Loss: 1.9731 | Train Acc: 25.37%
Epoch 07/10 | Loss: 1.9511 | Train Acc: 26.49%
Epoch 08/10 | Loss: 1.9017 | Train Acc: 28.30%
Epoch 09/10 | Loss: 1.8867 | Train Acc: 29.64%
Epoch 10/10 | Loss: 1.8723 | Train Acc: 30.05%
Option A finished in 45.69s


## 3. Train Option B: Standard Batched Model + FlashAttention (SDPA)

We train the refactored, native batched model. Optimizations applied:
* Batch size scaled to 512
* `num_workers=4`, `pin_memory=True`
* PyTorch's native `scaled_dot_product_attention` (SDPA) which dispatches to FlashAttention-2
* TensorFloat-32 (TF32) enabled for PyTorch matrix multiplications
* `torch.compile()` for kernel fusion
* Fused AdamW optimizer
* Automatic Mixed Precision (AMP) in `bfloat16`
* `zero_grad(set_to_none=True)`

In [6]:
# Enable TF32 for extra matmul speedup
torch.set_float32_matmul_precision('high')

epochs = 50

model_batched = BatchedVisualTransformer(
    img_size=32, patch_size=4, in_channels=3, num_classes=10,
    embed_dim=192, depth=6, num_heads=6, head_dim=32, mlp_ratio=2.0
).to(device)

# Configure compiler settings to avoid autotuning freezes on some devices
if os.environ.get("DISABLE_COMPILE", "0") == "1":
    print("Compilation disabled via DISABLE_COMPILE=1 environment variable.")
    model_batched_compiled = model_batched
else:
    try:
        import torch._inductor.config as inductor_config
        # Disable intensive autotuning to prevent compiler freezes on lower SM or VM GPUs
        inductor_config.max_autotune = False
        inductor_config.max_autotune_gemm = False
    except Exception:
        pass
    print("Compiling model (this might take a minute on the first epoch)...")
    model_batched_compiled = torch.compile(model_batched)

optimizer = optim.AdamW(model_batched_compiled.parameters(), lr=3.0e-3, weight_decay=5e-2, fused=True)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
criterion = nn.CrossEntropyLoss()

print("Training Option B (Standard Batched + FlashAttention + Compile)...")
start_batched = time.time()

for epoch in range(epochs):
    model_batched_compiled.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model_batched_compiled(inputs)
            loss = criterion(outputs, targets)
            
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    scheduler.step()
    train_acc = 100. * correct / total
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")
    
batched_total_time = time.time() - start_batched
print(f"Option B finished in {batched_total_time:.2f}s")

Compiling model (this might take a minute on the first epoch)...
Training Option B (Standard Batched + FlashAttention + Compile)...
Epoch 01/50 | Loss: 2.1366 | Train Acc: 18.29%
Epoch 02/50 | Loss: 2.0443 | Train Acc: 21.26%
Epoch 03/50 | Loss: 2.0356 | Train Acc: 22.27%
Epoch 04/50 | Loss: 1.9882 | Train Acc: 24.17%
Epoch 05/50 | Loss: 1.9721 | Train Acc: 24.20%
Epoch 06/50 | Loss: 1.9152 | Train Acc: 26.35%
Epoch 07/50 | Loss: 1.8965 | Train Acc: 28.43%
Epoch 08/50 | Loss: 1.8796 | Train Acc: 28.62%
Epoch 09/50 | Loss: 1.8096 | Train Acc: 31.78%
Epoch 10/50 | Loss: 1.7786 | Train Acc: 31.96%
Epoch 11/50 | Loss: 1.7432 | Train Acc: 33.93%
Epoch 12/50 | Loss: 1.7141 | Train Acc: 35.19%
Epoch 13/50 | Loss: 1.6856 | Train Acc: 36.79%
Epoch 14/50 | Loss: 1.6513 | Train Acc: 37.96%
Epoch 15/50 | Loss: 1.6452 | Train Acc: 37.89%
Epoch 16/50 | Loss: 1.6094 | Train Acc: 40.30%
Epoch 17/50 | Loss: 1.6045 | Train Acc: 39.72%
Epoch 18/50 | Loss: 1.5422 | Train Acc: 43.07%
Epoch 19/50 | Loss: 1.

## 4. Speed Comparison

In [5]:
speedup = vmap_total_time / batched_total_time
print(f"Option A (Optimized vmap): {vmap_total_time:.2f}s")
print(f"Option B (Batched + SDPA + Compile): {batched_total_time:.2f}s")
print(f"Speedup Factor: {speedup:.2fx} faster! (Note: Option B includes ~20-30s first-epoch compile overhead)")

Option A (Optimized vmap): 45.69s
Option B (Batched + SDPA + Compile): 89.20s


ValueError: Invalid format specifier '.2fx' for object of type 'float'

## 5. Visualizing Attention Maps & Positional Embeddings

We visualize the attention maps and positional similarity. The batched model `model_batched` is fully backward-compatible. In `eval()` mode, it automatically computes manual attention and saves the `attn_weights` in the exact shape (`[Hn, Sl, Sl]`) required by this visualization code.

In [ ]:
# Set batched model to evaluation mode
model_batched.eval()

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
fig, axs = plt.subplots(3, 4, figsize=(12, 9))

# Positional embeddings: shape is [1, num_patches + 1, embed_dim]
pos_embed = model_batched.patch_embed.pos_embed.detach().cpu()
spatial_pos = pos_embed[0, 1:]

norm_pos = spatial_pos / spatial_pos.norm(dim=-1, keepdim=True)
similarity = torch.mm(norm_pos, norm_pos.t()).numpy()

for i in range(3):
    img_tensor, label_idx = testset[i]
    
    with torch.no_grad():
        # Pass single image directly (shape [3, 32, 32]). The model handles it automatically!
        _ = model_batched(img_tensor.to(device))
        
    # Access attention weights from the batched multihead attention block
    attn = model_batched.blocks[-1].msa.attn_weights.detach().cpu()
    mean_attn = attn.mean(dim=0)
    cls_attn = mean_attn[0, 1:]
    
    cls_attn_grid = cls_attn.reshape(8, 8).numpy()
    cls_attn_resized = np.kron(cls_attn_grid, np.ones((4, 4)))
    
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.2023, 0.1994, 0.2010]) + np.array([0.4914, 0.4822, 0.4465])
    img_np = np.clip(img_np, 0, 1)
    
    axs[i, 0].imshow(img_np)
    axs[i, 0].set_title(f"Target: {classes[label_idx]}")
    axs[i, 0].axis('off')
    
    axs[i, 1].imshow(cls_attn_grid, cmap='viridis')
    axs[i, 1].set_title("CLS Attention (8x8)")
    axs[i, 1].axis('off')
    
    axs[i, 2].imshow(img_np)
    axs[i, 2].imshow(cls_attn_resized, cmap='jet', alpha=0.5)
    axs[i, 2].set_title("Attention Overlay")
    axs[i, 2].axis('off')
    
    center_similarity = similarity[28].reshape(8, 8)
    axs[i, 3].imshow(center_similarity, cmap='hot')
    axs[i, 3].set_title("Pos Sim (Center vs All)")
    axs[i, 3].axis('off')

plt.tight_layout()
plt.show()